# TriadLM M1 — base_50m validation on Kaggle free GPU

$0 budget. Before running:
1. **Accelerator ON**: Settings → Accelerator → GPU T4 x2 (free ~30h/week, resets weekly).
2. Push this repo to GitHub (free) and set `REPO_URL` below — or attach it as a Kaggle dataset.
3. Add your Hugging Face token as a Kaggle **Secret** named `HF_TOKEN` (free account) for checkpoint upload.
4. Sessions cap at ~12h: this run is ~15 min. If interrupted, re-run the train cell with `--resume` (see cell).

Honest framing: `corpus_v1` is only ~37k tokens, so this validates the M1 *pipeline*
(AMP, ckpts, resume, registry, eval) — it will overfit. A real M1 needs ~1000x more tokens
(Wikipedia dumps / OSCAR, free on Kaggle Datasets) + more Chichewa (JW300/Masakhane).

In [ ]:
!pip install -q torch tokenizers pyyaml tqdm fastapi uvicorn "pydantic>=2" requests huggingface_hub
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-ONLY — enable GPU!")

In [ ]:
REPO_URL = "https://github.com/PhillipMtalika/triadlm.git"
!git clone $REPO_URL triadlm 2>/dev/null || (cd triadlm && git pull)
%cd triadlm
!python -m pytest tests/test_tokenizer.py tests/test_model.py -q 2>&1 | tail -n 2

In [ ]:
import os
if not os.path.exists("data/checkpoints/base_50m/tokenizer/tokenizer.json"):
    from triadlm.tokenizer import TriadTokenizer
    import glob
    TriadTokenizer.train(sorted(glob.glob("data/raw/*.txt")), 8192,
                           "data/checkpoints/base_50m/tokenizer")
!python -m data.preprocess --config configs/kaggle_50m.yaml --corpus-name corpus_v1 --license "CC-BY-SA (Wikipedia) + project-internal (seed)"
!python -m data.quality_report --manifest data/manifests/corpus_v1.json

In [ ]:
# Fresh run (~15 min on T4). If interrupted, re-run with the --resume line instead.
!python -m triadlm.train --config configs/kaggle_50m.yaml
# !python -m triadlm.train --config configs/kaggle_50m.yaml --resume data/checkpoints/kaggle_50m/step_250.pt

In [ ]:
!python -m evals.run_eval --checkpoint data/checkpoints/kaggle_50m/final.pt --variant base --out experiments/runs/kaggle-50m.json
!head -n 5 experiments/registry.csv

In [ ]:
# Push checkpoints + manifest to Hugging Face Hub (free, unlimited public repos).
from huggingface_hub import HfApi, create_repo
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
create_repo("triadlm-kaggle-50m", private=False, exist_ok=True, token=token)
api.upload_folder(folder_path="data/checkpoints/kaggle_50m", repo_id=api.whoami(token)["name"] + "/triadlm-kaggle-50m", token=token)
api.upload_file(path_or_fileobj="data/manifests/corpus_v1.json", path_in_repo="corpus_v1.json",
               repo_id=api.whoami(token)["name"] + "/triadlm-kaggle-50m", token=token)
print("uploaded")